In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q2-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split

# Load Labels
labels_df = pd.read_csv(os.path.join(path, "labels.csv"))
img_dir = os.path.join(path, "images")

images = []
ages = []

# Load Images and Resize
print("Loading images...")
for index, row in labels_df.iterrows():
    img_name = row.iloc[0]
    age = row.iloc[1]
    img_path = os.path.join(img_dir, img_name)

    if os.path.exists(img_path):
        img = Image.open(img_path).convert('RGB')
        img_array = np.array(img) / 255.0 # Normalize pixel values to [0, 1]
        images.append(img_array)
        ages.append(age)

X = np.array(images)
y = np.array(ages)

# Transpose image dimensions to match PyTorch format (N, C, H, W)
X = np.transpose(X, (0, 3, 1, 2))

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
X_train.dtype, y_train.dtype

In [ ]:
# 1. Convert Numpy arrays to PyTorch Tensors
import torch
from torchvision.transforms.functional import to_tensor

X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)




In [ ]:
# 2. Create TensorDataset objects

from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(
    X_train, y_train
)

test_dataset = TensorDataset(
    X_test, y_test
)




In [ ]:
# 3. Create DataLoaders

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2
)

In [ ]:
# 4. Print shape of one batch
X_train_batch, y_train_batch = next(iter(train_loader))
X_test_batch, y_test_batch = next(iter(test_loader))

print(f"Train Batch Shape for X: {X_train_batch.shape}, and for y : {y_train_batch.shape}")

print(f"Test Batch Shape for X: {X_test_batch.shape}, and for y: {y_test_batch.shape}")

In [ ]:
# 5. Display sample images

import matplotlib.pyplot as plt

images, labels = next(iter(train_loader))

# Display the first 6 images in the batch
plt.figure(figsize=(8, 4))

for i in range(6):
    plt.subplot(2, 3, i + 1)
    plt.imshow(images[i].squeeze()[2], cmap='gray')
    plt.title(f"Label: {labels[i].item()}")
    plt.axis('off')

plt.tight_layout()
plt.show()


In [ ]:
# Task 1: Write your model class here:
import torch.nn as nn

class NN4Layer(nn.Module):
  def __init__(self, input_dim, hidden_dim, output_dim):
    super(NN4Layer, self).__init__()

    self.layer1 = nn.Linear(input_dim, hidden_dim)
    self.layer2 = nn.Linear(hidden_dim, hidden_dim)
    self.layer3 = nn.Linear(hidden_dim, hidden_dim)
    self.layer4 = nn.Linear(hidden_dim, output_dim)
    self.relu = nn.ReLU()

  def forward(self, x):
    z1 = self.layer1(x)
    a1 = self.relu(z1)

    z2 = self.layer2(a1)
    a2 = self.relu(z2)

    z3 = self.layer3(a2)
    a3 = self.relu(z3)

    output = self.layer4(a3)

    return output

In [ ]:
# Task 2: Write your training loop here:

def train_one_epoch(model, train_loader, optimizer, criterion, device):
  model.train()

  epoch_loss = 0.0

  for X_batch, y_batch in train_loader:
    X_batch = X_batch.flatten(start_dim=1).to(device)
    y_batch = y_batch.view(-1, 1).to(device)

    # Forward Propagation

    outputs = model(X_batch)
    loss = criterion(outputs, y_batch)

    # Backward Propagation

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    epoch_loss += loss.item()
  avg_loss = epoch_loss/len(train_loader)
  return avg_loss

In [ ]:
# Task 3: Write your validation loop here:

def validate(model, test_loader, criterion, device):

  model.eval()

  with torch.no_grad():
    val_loss = 0.0
    for X_batch, y_batch in test_loader:

      X_batch = X_batch.flatten(start_dim=1).to(device)
      y_batch = y_batch.view(-1, 1).to(device)

      outputs = model(X_batch)
      loss = criterion(outputs, y_batch)

      val_loss += loss.item()

  avg_loss = val_loss/len(test_loader)
  return avg_loss


In [ ]:
# Task 4: Define device, model, loss, optimizer:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

input_dim = 3*36*36
hidden_dim=30
output_dim=1

model = NN4Layer(input_dim, hidden_dim, output_dim)

criterion = nn.MSELoss()

learning_rate = 0.001
optimizer = torch.optim.AdamW(model.parameters(), lr = learning_rate, weight_decay=.09)

In [ ]:
# Task 5: Start training for 20 epochs:
NUM_EPOCHS = 20

val_losses = []
train_losses = []

for epoch in range(NUM_EPOCHS):
  train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)

  val_loss = validate(model, test_loader, criterion, device)

  val_losses.append(val_loss)

  train_losses.append(train_loss)

  print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}\n")
  print(f"Training Loss : {train_loss}, Validation Loss: {val_loss}\n")

In [ ]:
# Task 1: Write your code here:
plt.plot(train_losses, label="TRAINING", color='blue')
plt.plot(val_losses, label="VALIDATION", color='orange')
plt.xlabel("EPOCH")
plt.ylabel("MSE Loss")
plt.grid()
plt.legend()

In [ ]:
# Task 2 (Bonus): Write your code here:
# Set model to evaluation mode
model.eval()
# Get one batch from the test DataLoader
images, labels = next(iter(test_loader))
# Move images to device
images = images.to(device)
labels = labels.to(device)

with torch.no_grad():
    # Flatten images before passing to the model
    outputs = model(images.view(images.size(0), -1))

# Move tensors back to CPU for plotting
images = images.cpu()
outputs = outputs.cpu()
labels = labels.cpu()

# Plot first 6 predictions
plt.figure(figsize=(8, 4))
for i in range(6):
    plt.subplot(2, 3, i + 1)
    plt.imshow(images[i].squeeze()[0], cmap='gray')
    plt.title(f"True: {labels[i]} | Pred: {int(outputs.squeeze()[i])}")
    plt.axis('off')

plt.tight_layout()
plt.show()